# Main Panel Construction

This notebook constructs the universe in the research project. I've downloaded the US stocks data from https://stooq.com/db/h/.

The structure of the notebook is as follows:
- Load intraday ETFs data and reformat them to make the panel.
- Calculate the releavant quantities needed, *e.g.*, cumulative 3-bar returns, 78-bar realised volatility, and *etc*.
- Save to a parquet file.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## Define useful functions to read stooq data from folders

In [2]:
# only the ETFs that are used in the project is present
def find_stooq_intraday_file(ticker, root_directory="./data/5 min/us"):
    root = Path(root_directory)
    target = f"{ticker.lower()}.us.txt"

    matches = list(root.rglob(target))

    if len(matches) == 0:
        return None
    
    return matches[0]

In [3]:
def _read_stooq_intraday_file(file_path, ticker=None):
    if ticker is None: raise ValueError("Enter ticker.")

    df = pd.read_csv(file_path)
    df.columns = [c.strip().lower() for c in df.columns]

    # rename map for relevant columns
    rename_map = {
        "<ticker>": "ticker_raw",
        "<date>": "date_raw",
        "<time>": "time_raw",
        "<close>": "close"
    }
    df = df.rename(columns=rename_map)

    required = ["date_raw", "time_raw", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{file_path} missing required columns: {missing}")
    
    # combine the date and time to a date-time column
    dt_str = df["date_raw"].astype(str).str.zfill(8) + df["time_raw"].astype(str).str.zfill(6)
    df["date_time"] = pd.to_datetime(dt_str, format="%Y%m%d%H%M%S", errors="coerce")

    # convert from Polish time to EST, takes care of daylight saving mismatch
    df["date_time"] = df["date_time"].dt.tz_localize("Europe/Warsaw").dt.tz_convert("America/New_York").dt.tz_localize(None)
    df["date"] = df["date_time"].dt.date
    df["time"] = df["date_time"].dt.time

    # use this to bucket rough time of the day, e.g., morning, afternoon, ...
    df["minute_of_day"] = df["date_time"].dt.hour * 60 + df["date_time"].dt.minute

    # keep regular trading hours: 09:30 to 16:00
    df = df[(df["minute_of_day"] >= 570) & (df["minute_of_day"] <= 960)].copy()

    # calculate the returns and log returns
    df["close_num"] = pd.to_numeric(df["close"])
    df["ret"] = df["close_num"].pct_change()
    df["log_ret"] = np.log(df["close_num"]).diff()

    out = pd.DataFrame(
        {
            "ticker": ticker,
            "date_time": df["date_time"],
            "date": df["date"],
            "time": df["time"],
            "minute_of_day": df["minute_of_day"],
            "close": df["close_num"],
            "ret": df["ret"],
            "log_ret": df["log_ret"]
        }
    )

    return out

def make_panel(tickers, root_directory="./data/5 min/us"):
    panels = []
    missing = []

    for t in tickers:
        file_path = find_stooq_intraday_file(t, root_directory=root_directory)
        if file_path is None:
            missing.append(t.upper())
            continue

        panels.append(_read_stooq_intraday_file(file_path, t))

    if not panels:
        raise FileNotFoundError(f"No requested files found under {root_directory}. Missing={missing}")
    
    # concat all the individual ticker data
    panel = (
        pd.concat(panels, ignore_index=True)
        .sort_values(["ticker", "date_time"])
        .reset_index(drop=True)
    )

    return panel, missing

## Make the panel and check daylight savings time alignment

In [4]:
# 12 tickers used in the project
tickers = ["SPY", "QQQ", "TLT", "GLD", "HYG", "IWM", "DIA", "EEM", "IEF", "LQD", "XLE", "XLF"]

panel, missing = make_panel(tickers=tickers, root_directory="data/5 min/us")
panel.head()

,ticker,date_time,date,time,minute_of_day,close,ret,log_ret
0,DIA,2025-10-27 09:30:00,2025-10-27,09:30:00,570,474.68,NaN,NaN
1,DIA,2025-10-27 09:35:00,2025-10-27,09:35:00,575,474.34,-0.000716,-0.000717
2,DIA,2025-10-27 09:40:00,2025-10-27,09:40:00,580,474.15,-0.000401,-0.000401
3,DIA,2025-10-27 09:45:00,2025-10-27,09:45:00,585,474.07,-0.000169,-0.000169
4,DIA,2025-10-27 09:50:00,2025-10-27,09:50:00,590,474.47,0.000844,0.000843


In [5]:
# check the relevant daylight saving dates, note stooq data only from 2025-10-27
check_dates = pd.to_datetime(["2025-10-27", "2025-11-03"]).date

session_check = (
    panel.groupby(["ticker", "date"])
    .agg(
        first_time=("time", "min"),
        last_time=("time", "max"),
        n_bars=("time", "size"),
    )
    .reset_index()
)

session_check_on_dates = session_check[session_check["date"].isin(check_dates)]
session_check_on_dates

,ticker,date,first_time,last_time,n_bars
0,DIA,2025-10-27,09:30:00,15:55:00,78
5,DIA,2025-11-03,09:30:00,15:55:00,78
100,EEM,2025-10-27,09:30:00,15:55:00,78
105,EEM,2025-11-03,09:30:00,15:55:00,78
200,GLD,2025-10-27,09:30:00,15:55:00,78
205,GLD,2025-11-03,09:30:00,15:55:00,78
300,HYG,2025-10-27,09:30:00,15:55:00,78
305,HYG,2025-11-03,09:30:00,15:55:00,78
400,IEF,2025-10-27,09:30:00,15:55:00,78
405,IEF,2025-11-03,09:30:00,15:55:00,78


In [6]:
panel = panel.sort_values(["ticker", "date_time"]).copy()

# bar count for each day
panel["bar_idx"] = panel.groupby(["ticker", "date"]).cumcount()

# rolling 3 bar (15 mins) return. same day only
panel["ret_3b"] = (
    panel.groupby(["ticker", "date"])["log_ret"]
    .transform(lambda s: s.rolling(3, min_periods=3).sum())
)

# rolling 78 bar (one trading day) realised vol. can be different days
panel["vol_78b"] = (
    panel.groupby("ticker")["log_ret"]
    .transform(lambda s: s.rolling(78, min_periods=78).std())
)

# z-score of the 3 bar return standardised
panel["z_shock_3b"] = panel["ret_3b"] / (panel["vol_78b"] * np.sqrt(3))

# forward cumulative returns, same day only
for h in [1, 3, 6, 12]:
    panel[f"ret_fwd_{h}b"] = (
        panel.groupby(["ticker", "date"])["log_ret"]
        .transform(lambda s: s.shift(-1).rolling(h, min_periods=h).sum())
    )

panel.tail()

,ticker,date_time,date,time,minute_of_day,close,ret,log_ret,bar_idx,ret_3b,vol_78b,z_shock_3b,ret_fwd_1b,ret_fwd_3b,ret_fwd_6b,ret_fwd_12b
92616,XLF,2026-03-20 15:35:00,2026-03-20,15:35:00,935,48.818,-0.001248,-0.001249,73,-0.003313,0.001096,-1.745083,0.003109,-0.000510,-0.001021,-0.000714
92617,XLF,2026-03-20 15:40:00,2026-03-20,15:40:00,940,48.970,0.003114,0.003109,74,-0.000510,0.001153,-0.255563,-0.001328,0.000532,-0.002757,-0.001634
92618,XLF,2026-03-20 15:45:00,2026-03-20,15:45:00,945,48.905,-0.001327,-0.001328,75,0.000532,0.001162,0.264239,0.004183,0.005964,0.002651,0.002446
92619,XLF,2026-03-20 15:50:00,2026-03-20,15:50:00,950,49.110,0.004192,0.004183,76,0.005964,0.001251,2.752925,-0.000204,0.002651,0.002141,0.002039
92620,XLF,2026-03-20 15:55:00,2026-03-20,15:55:00,955,49.100,-0.000204,-0.000204,77,0.002651,0.001251,1.223652,NaN,NaN,NaN,NaN


## Save to parquet

In [ ]:
panel.to_parquet("./outputs/parquets/research_panel.parquet")